# aw_04_a2 — Stage A2: PlayWorld DPO on verifier-mined pairs (Track A, RQ2)

**Protocol**: §5.1 A2, §5.2 E-RANDPAIR control (mined later at equal pair count).
**Parent**: A1 adapter `20260801-030335--a1-playworld-sft--s42--e24d72` (sha256 pinned in the recipe — lineage-verified at load).

Pipeline: sample K=8 candidates/prompt from the A1 policy (temperature 0.8, canonical
conditioning incl. opener seed) → score with the frozen hybrid verifier →
margin-gated chosen/rejected mining (`hybrid_verifier_rank`, margin ≥ 0.10) →
DPO (LR 5e-7, 1 epoch) → eval on the frozen suites → paired analysis A2 vs A1.

**Note**: A2 uses a NEW artifacts repo `m97j/aw-runs-a2` (one training run per repo);
eval runs still nest under `runs/` inside it.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title a_a2_data — mine verifier-guided preference pairs from the A1 policy
!python scripts/build_training_data.py
!python scripts/build_eval_suites.py --episodes-per-suite 300

A1_RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_RUN_ID}
print("\n".join(out))
adapter_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/mine_playworld_pairs.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir} \
  --prompt-file data/train/playworld_prompts.jsonl \
  --num-candidates 8 --temperature 0.8 --batch-size 100 \
  --selection-method hybrid_verifier_rank --minimum-margin 0.10 \
  --output data/train/playworld_preference.jsonl \
  --hf-sync-repo m97j/aw-playworld


In [ ]:
# @title b_a2_train — DPO from the A1 parent (lineage-verified)
!python scripts/run_experiment.py \
  --config configs/experiments/a2_playworld_dpo.yaml \
  --parent-adapter-dir {adapter_dir} \
  --override data.source.local_path=data/train/playworld_preference.jsonl \
  --hf-sync-repo m97j/aw-runs-a2


In [ ]:
# @title c_a2_eval — A2 adapter, canonical profile
A2_RUN_ID = ""  # <- fill from b_a2_train output ("run_id: ...")
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_RUN_ID}
print("\n".join(out))
a2_adapter_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {a2_adapter_dir} \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a2


In [ ]:
# @title f_a2_analysis — A2 vs A1 (paired), the RQ2 primary readout
A2_EVAL_RUN = ""  # <- fill from c_a2_eval output ("eval run: ...")
A1_EVAL_RUN = "20260801-063425--eval-playworld--s42--3bf440"

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {A1_EVAL_RUN} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a2 --run-id {A2_EVAL_RUN} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{A2_EVAL_RUN} --label-a a2-dpo \
  --run-b runs/{A1_EVAL_RUN} --label-b a1-sft \
  --output runs/{A2_EVAL_RUN}/analysis_vs_a1.json \
  --hf-sync-repo m97j/aw-runs-a2


## Stage checklist
- [ ] mining manifest: pairs_accepted / decision_counts sane (report both)
- [ ] DPO run completed, artifacts on the Hub, lineage verified
- [ ] A2 vs A1: legality + state-consistency deltas are the H2 readout
- [ ] Next: E-RANDPAIR control at EQUAL pair count (aw_10) uses the same mining
      script with `--selection-method random_pairing`
